# Verifica se un albero binario è di ricerca

In [2]:
from typing import override

from alberi.alberibinari import AlberoBin

class abr(AlberoBin):
    def diRicerca(self):
        return abr._diRicercaRic(self, float('-inf'), float('inf'))
    @staticmethod
    def _diRicercaRic(nodo, min:int, max:int):
        if nodo is None:
            return True
        if nodo.val <= min or nodo.val >= max:
            return False
        return abr._diRicercaRic(nodo.sin, min, nodo.val) and abr._diRicercaRic(nodo.des.val, max)

# Dato un albero binario verificare se è bilanciato
con un algoritmo ricorsivo di complessità lineare

In [3]:
class abb(AlberoBin):
    def is_bilanciato(self):
        return abb._is_bilanciato(self)[1]
    @staticmethod
    def _is_bilanciato(nodo:AlberoBin) -> tuple[int, bool]:
        if nodo is None:
            return (0, True)
        resSin = abb._is_bilanciato(nodo.sin)
        resDes = abb._is_bilanciato(nodo.des)
        if (not resDes[1] or not resSin[1]):
            return (0, False)
        return (max(resSin[0], resDes[0])+1, abs(resSin[0] - resDes[0]) <= 1)

    # versione incredibilmente pulita
    def is_bilanciato2(self):
        try:
            abb._is_bilanciato2(self)
            return True
        except ValueError:
            return False
    @staticmethod
    def _is_bilanciato2(nodo:AlberoBin) -> int:
        if nodo is None:
            return 0
        hSin = abb._is_bilanciato2(nodo.sin)
        hDes = abb._is_bilanciato2(nodo.des)
        if abs(hSin-hDes) > 1:
            raise ValueError("non bilanciato")
        return max(hSin, hDes) + 1

# Visita sui grafi

In [4]:
from grafi.grafi import GrafoMA
from grafi.grafi import GrafoLA
from grafi.grafino import GrafoLANO
from grafi.grafino import GrafoMANO

class Visita(GrafoLA):
    def _visita_prof(self, nodo: int, visitati: list[bool], res: list[int]):
        if not visitati[nodo]:
            visitati[nodo] = True
            res.append(nodo)
            for a in self.adiacenti(nodo):
                self._visita_prof(a, visitati, res)
    def visita_prof(self, nodo: int) -> list[int]:
        res: list[int] = []
        visitati: list[bool] = [False for _ in range(self.n)]
        self._visita_prof(nodo, visitati, res)
        return res

    def visita_vent(self, nodo:int) -> list[int]:
        res: list[int] = [nodo]
        visitati: list[bool] = [False for _ in range(self.n)]
        coda: list[int] = [nodo]
        visitati[nodo] = True
        while(coda):
            curr: int = coda.pop(0)
            for a in self.adiacenti(curr):
                if not visitati[a]:
                    visitati[a] = True
                    coda.append(a)
                    res.append(a)
        return res

# Verifica se un grafo è un albero / verifica aciclicità

In [5]:
from grafi.grafi import GrafoMA
from grafi.grafi import GrafoLA
from grafi.grafino import GrafoLANO
from grafi.grafino import GrafoMANO

class GrafoAlberoOR(Visita):
    def verificaAlberoOR(self):
        return self.n == self.m + 1 and self.econnessoOR(0)
    def econnessoOR(self, nodo: int) -> bool:
        visita = self.visita_vent(nodo)
        if len(visita) == self.n:
            return True
        return False
    def eaciclicoOR(self):
        visitati: list[bool] = [False for _ in range(self.n)]
        k: int = 0
        for i in range(self.n):
            if not visitati[i]:
                k += 1
                self._visita_prof(i, visitati, [])
        return self.n < self.m + k

class GrafoAlberoNO(GrafoLANO):
    def aciclicoNO(self):
        rimossi: list[bool] = [False for _ in range(self.n)]
        gradi: list[int] = [0 for _ in range(self.n)]
        for a in self.archi():
            gradi[a[1]] += 1
        curr = self.trovaZero(gradi, rimossi)
        while curr != -1:
            rimossi[curr] = True
            for a in self.adiacenti(curr):
                gradi[a] -= 1
            curr = self.trovaZero(gradi, rimossi)
        return self.tuttiZeri(gradi)
    def tuttiZeri(self, gradi: list[int]):
        for i in gradi:
            if i != 0:
                return False
        return True
    def trovaZero(self, gradi: list[int], rimossi: list[bool]):
        for i in range(len(gradi)):
            if gradi[i] == 0 and not rimossi[i]:
                return i
        return -1

# Scheduling delle attività
Caso pratico di utilizzo di un grafo aciclico orientato (DAG).

- Ogni nodo del grafo rappresenta un'attività da svolgere.
- Ogni arco rappresenta una relazione di propedeuticità tra due attività (es. A→B significa "A deve terminare prima che B possa iniziare").
- Ogni attività ha un proprio tempo di esecuzione.
- Hai a disposizione un numero illimitato di esecutori (quindi più attività possono partire in parallelo, appena le loro propedeuticità sono soddisfatte).
- Si parte al tempo 0.

Domanda: qual è il tempo minimo necessario per completare tutte le attività, rispettando i vincoli di propedeuticità?

In [6]:
class Scheduler(GrafoAlberoOR):
    def scheduling(self) -> int:
        t: int = 0
        visitati: list[bool] = [False for _ in range(self.n)]
        gradi: list[int] = [0 for _ in range(self.n)]
        for p, a in self.archi():
            gradi[a] += 1
        esec: list[tuple[int, int]] = []
        esec.extend(self.trovazeri(gradi, visitati))
        while esec:
            curr = min(esec, key = lambda x: x[1])
            esec.remove(curr)
            t += curr[1]
            for i in range(len(esec)):
                esec[i] = (esec[i][0], esec[i][1] - curr[1])
            for a in self.adiacenti(curr[0]):
                gradi[a] -= 1
            esec.extend(self.trovazeri(gradi, visitati))
        return t

    # trova tutti quelli che hanno grado di ingresso 0, li aggiunge a una lista associandoci il tempo rimanente alla fine dell'esecuzione e li restituisce
    def trovazeri(self, gradi: list[int], visitati: list[bool]) -> list[tuple[int, int]]:
        res: list[tuple[int, int]] = []
        for i in range(len(gradi)):
            if gradi[i] == 0 and not visitati[i]:
                visitati[i] = True
                res.append((i,i))
        return res

In [7]:
from grafi.grafi import GrafoLA
from grafi.grafi import GrafoMA

# Scheduler, Visita e GrafoAlberoOR li presumo già definiti in celle precedenti del notebook


def costruisci_dag() -> Scheduler:
    # 5 nodi, indice == durata (nodo 0 dura 0, nodo 3 dura 3, ecc.)
    g = Scheduler(5)
    g.aggiungiarco(0, 1)
    g.aggiungiarco(0, 2)
    g.aggiungiarco(1, 3)
    g.aggiungiarco(2, 3)
    g.aggiungiarco(3, 4)
    return g


def main():
    g = costruisci_dag()

    # Cammino critico atteso: 0 -> 2 -> 3 -> 4 = durate 0+2+3+4 = 9
    # (l'altro ramo 0 -> 1 -> 3 -> 4 = 0+1+3+4 = 8, quindi non è quello critico)
    atteso = 9
    risultato = g.scheduling()

    print(f"Tempo minimo calcolato: {risultato}")
    print(f"Atteso: {atteso}")
    assert risultato == atteso, f"FALLITO: atteso {atteso}, ottenuto {risultato}"
    print("Test superato.")


if __name__ == "__main__":
    main()

Tempo minimo calcolato: 9
Atteso: 9
Test superato.


# Backtracking
## N regine
Date N regine e una scacchiera N×N, posiziona tutte le N regine sulla scacchiera in modo che nessuna coppia di regine si attacchi a vicenda.
Vincoli: due regine si attaccano se si trovano sulla stessa riga, stessa colonna, o stessa diagonale (principale o secondaria).
Task tipico da esercizio:

1. Trovare una soluzione (una qualsiasi disposizione valida)
2. Trovare tutte le soluzioni
3. Contare il numero totale di soluzioni

Traccia consigliata (stile backtracking):

Rappresenta la soluzione come un array pos[] di lunghezza N, dove pos[i] = colonna in cui si trova la regina della riga i (questo elimina automaticamente il conflitto "stessa riga")
Costruisci la soluzione riga per riga: alla riga i, prova ogni colonna c da 0 a N-1
Prima di piazzare, verifica compatibilità con le regine già piazzate nelle righe precedenti (colonna diversa, diagonali diverse: |pos[i] - pos[j]| != |i - j|)
Se compatibile, piazza e ricorsione sulla riga successiva; altrimenti backtrack

In [2]:
from backtracking.backtracking import ProblemaBack

class Nregine(ProblemaBack):
    N = 9
    pos: list[int] = [-1 for _ in range(N)] # i indice di riga, pos[i] indice di colonna


    def primaScelta(self, liv: int):
        self.pos[liv] = 0
        return True

    def successivaScelta(self, liv: int)->bool:
        if self.pos[liv] >= self.N-1:
            return False
        self.pos[liv] += 1
        return True

    def solCompleta(self, liv: int)->bool:
        for i in range(self.N):
            if self.pos[i] == -1:
                return False
        return True

    def verificaVincoli(self, liv: int)->bool:
        for i in range(liv):
            if self.pos[i] == self.pos[liv] or abs(self.pos[i] - self.pos[liv]) == abs(i-liv):
                return False
        return True

    def costruisciSoluzione(self, liv: int):
        sol: list[list[int]] = [[0 for _ in range(self.N)] for _ in range(self.N)]
        for i in self.pos:
            sol[i][self.pos[i]] = 1
        for i in range(self.N):
            for j in range(self.N):
                print(sol[i][j], end="")
            print()

def main():
    r = Nregine().risolvi()

if __name__ == "__main__":
    main()

100000000
001000000
000001000
000000010
010000000
000100000
000000001
000000100
000010000


# Pseudocodici algoritmi noti
## Prim

In [ ]:
res = nodo1
taglio: heap = {G.n}
taglio.aggiungiT(nodo1) # aggiunge al taglio una coppia nodo-peso per tutti gli adiacenti di nodo
while res.n < G.n: # alb.n numero di nodi dell'albero e G.n numero di nodi del grafo
    nuovo = estrai_min(taglio) # estrae la coppia nodo-peso con peso minore e la rimuove
    res.add(nuovo) # aggiunge all'albero il nuovo nodo e il relativo arco
    taglio.aggiungiT(nuovo[1]) # aggiunge al taglio la coppia nodo-peso di tutti gli adiacenti, se è già presente il nodo verifica se il nuovo arco ha un costo minore
return res

# Esercizio 1
Si consideri una classe `AlberoB` che rappresenta alberi binari in cui la parte informativa di ogni nodo è un numero intero. Si assuma che in tale classe siano implementati i seguenti metodi:
```
public interface AlberoB {
/* restituisce il sottoalbero destro dell’albero corrente, la complessità temporale è (1)*/
public AlberoB destro( );
/* restituisce il sottoalbero sinistro dell’albero corrente, la complessità temporale è (1)*/
public AlberoB sinistro( );
/* restituisce il valore memorizzato nella radice dell’albero, la complessità temporale è (1)*/
public int val( );
}
```
Si deve realizzare un metodo `public static boolean eRipetuto (AlberoB a, int x) {…}` che restituisce `true` se e solo se vi è almeno un nodo `n` nell’albero a tale che l’intero `x` appare sia nel sottoalbero sinistro che nel sottoalbero destro di `n`.

In [8]:
from alberi.alberibinari import AlberoBin

class es1(AlberoBin):
    def eRipetuto(self, nodo: AlberoBin, x: int) -> bool:
        if nodo is None:
            return False
        return (es1.ricerca(nodo.des, x) and es1.ricerca(nodo.sin, x)) or self.eRipetuto(nodo.des, x) or self.eRipetuto(nodo.sin, x)
    @staticmethod
    def ricerca(nodo: AlberoBin, x: int):
        if nodo is None:
            return False
        return nodo.val == x or es1.ricerca(nodo.des, x) or es1.ricerca(nodo.sin, x)

### Complessità
**Caso migliore**

Nel caso migliore `x` è figlio della radice dell'albero, e la complessità sarà quindi costante, sia spaziale che temporale $T_m, S_m \in \Theta(1)$

**Caso peggiore**

Nel caso peggiore l'albero è degenere, l'algoritmo scorre tutti i nodi in `eRipetuto()` e per ogni nodo lancia una chiamata a `ricerca()` che a sua volta effettua una visita, comportando una complessità temporale $T_p \in \Theta(n^2)$. La spaziale invece è lineare in quanto il numero di chiamate contemporaneamente sullo stack è pari al numero di nodi $S_p \in \Theta(n)$

# Esercizio 2
verificare `almenoUnDoppio(a: AlberoBin, x: int)` che restituisce `True` se e solo se esiste `x` che appare in un nodo non foglia e appare in un nodo foglia di `a`

In [9]:
def almenoUnDoppio(a: AlberoBin, x: int) -> bool:
    return ricFoglia(a, x) and ricMid(a, x)
def ricFoglia(a, x: int) -> bool:
    if a is None:
        return False
    if a.des is None and a.sin is None:
        return a.val == x
    return ricFoglia(a.sin, x) or ricFoglia(a.des, x)
def ricMid(a, x: int) -> bool:
    if a is None or (a.des is None and a.sin is None):
        return False
    return a.val == x or ricMid(a.sin, x) or ricMid(a.des, x)

### Complessità
**Caso migliore**

Nel caso migliore il figlio sinistro della radice è una foglia e contiene `x`, come la radice. Le chiamate ricorsive non dipendono dalla grandezza dell'input in quanto non facciamo ipotesi sulla forma del sottoalbero destro: $T_m(n) = S_m(n) \in \Theta(1)$

**Caso peggiore**

Nel caso peggiore l'albero è degenere e la sua esplorazione comporterà una complessità $T_p(n) = S_p(n) \in \Theta(n)$

# Esercizio 3
`foglieM(a: AlberoBin)` restituisce `True` se e solo se esiste solo una foglia dell'albero che ha valore $\geq 0$

In [10]:
def foglieM(a:AlberoBin) -> bool:
    return conta(a) == 1
def conta(a) -> int:
    if a is None:
        return 0
    if a.des is None and a.sin is None:
        if a.val >= 0:
            return 1
        return 0
    return conta(a.sin) + conta(a.des)

### Complessità
Per produrre l'output l'algoritmo legge tutte le foglie quindi la complessità temporale, indipendentemente dal caso, è $T_m(n) = T_p(n) \in \Theta(n)$.

La complessità spaziale invece sarà lineare nel caso peggiore se l'albero è degenere e logaritmica nel caso migliore se l'albero è bilanciato $S_m(n) \in \Theta(log_2(n))$ e $S_p(n) \in \Theta(n)$

# Esercizio 4
Restituisci il livello del nodo con valore `x`, se non è presente restituisce `0`

In [11]:
def livelloX(a, x: int) -> int:
    return calcolo(a, x, 0)
def calcolo(a, x: int, curr: int) -> int:
    if a is None:
        return 0
    if a.val == x:
        return curr
    return max(calcolo(a.sin, x, curr + 1), calcolo(a.des, x, curr + 1))

calcola l'altezza del nodo con valore `x`, se non è presente restituisce `0`

In [12]:
def altezzaX(a, x: int) -> int:
    if a is None:
        return 0
    if a.val == x:
        return calcoloH(a, x, 0)
    return max(altezzaX(a.sin, x), altezzaX(a.des, x))
def calcoloH(a, x: int, curr: int) -> int:
    if a is None:
        return curr
    return max(calcoloH(a.sin, x, curr + 1), calcoloH(a.des, x, curr + 1))

### Complessità
**Temporale**

In entrambi gli algoritmi a prescindere dall'output verrà esplorato l'albero nella sua interezza, di conseguenza $T(n) \in \Theta(n)$.

**Spaziale**

La spaziale invece dipende dal numero di chiamate contemporaneamente sullo stack, che nel caso migliore, in un albero bilanciato sarà dato da $S_m(n) \in \Theta(log_2(n))$ , mentre nel caso peggiore, in un albero degenere tutte le chiamate sono sullo stack $S_p(n) \in \Theta(n)$